# Group 5 Capstone: Full Responsible ML Pipeline

This notebook provides the full HMDA pipeline in one place, organized section by section for easy reading, execution, and integration.

It implements:
- Data loading and validation
- Binary label handling
- Train/validation/test split
- Baseline and comparison model training
- Performance, fairness, intersectional, and robustness diagnostics
- Transparency analysis
- Artifact export (tables, figures, and model files)

## 1) Imports and Paths

In [2]:
from pathlib import Path
import sys
import json
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import RocCurveDisplay

project_root = Path.cwd()
if project_root.name == "notebooks":
    project_root = project_root.parent

# Ensure local project modules are importable from notebook execution context.
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.features.engineering import build_binary_target, prepare_modeling_frame
from src.models.train import build_models, make_splits, train_model
from src.models.fairness import fairness_by_group, intersectional_fairness
from src.models.robustness import drift_report, perturb_numeric_features
from src.models.transparency import logistic_coefficients, permutation_importance_table
from src.models.evaluate import classification_metrics

data_processed = project_root / "data" / "processed"
reports_tables = project_root / "reports" / "tables"
reports_figures = project_root / "reports" / "figures"
models_dir = project_root / "models"

for p in [data_processed, reports_tables, reports_figures, models_dir]:
    p.mkdir(parents=True, exist_ok=True)

input_file = data_processed / "hmda_lar_2024_cleaned.csv"
if not input_file.exists():
    raise FileNotFoundError(f"Missing cleaned dataset: {input_file}")

print(f"Project root: {project_root}")
print(f"Input file: {input_file}")

Project root: /Users/boratonaj/Documents/GWU_Study_Course/Spring_Class_2026/responsible_AI/Group_5_Capstone_Project
Input file: /Users/boratonaj/Documents/GWU_Study_Course/Spring_Class_2026/responsible_AI/Group_5_Capstone_Project/data/processed/hmda_lar_2024_cleaned.csv


## 2) Configuration
Set core run parameters here before executing the pipeline.

- `RANDOM_STATE`: reproducibility
- `USE_FULL_DATASET`: if `True`, training uses all rows (no sampling)
- `MAX_ROWS`: used only when `USE_FULL_DATASET=False`
- `MIN_GROUP_SIZE`: minimum subgroup size for fairness metrics
- `MAX_CATEGORICAL_CARDINALITY`: prevents one-hot explosion on high-cardinality columns
- `DROP_COLUMNS`: columns excluded from model features

In [17]:
RANDOM_STATE = 42
USE_FULL_DATASET = False  # Set True to train on all rows
MAX_ROWS = 100000  # Used only when USE_FULL_DATASET=False
MIN_GROUP_SIZE = 100
MAX_CATEGORICAL_CARDINALITY = 200

DROP_COLUMNS = [
    "lei",
    "activity_year",
    "submission_of_application",
]

# Resolve effective row cap based on mode
effective_max_rows = None if USE_FULL_DATASET else MAX_ROWS

# Simple guardrails
if not USE_FULL_DATASET and (effective_max_rows is None or effective_max_rows <= 0):
    raise ValueError("When USE_FULL_DATASET=False, MAX_ROWS must be a positive integer.")
if MIN_GROUP_SIZE < 1:
    raise ValueError("MIN_GROUP_SIZE must be >= 1.")
if MAX_CATEGORICAL_CARDINALITY < 2:
    raise ValueError("MAX_CATEGORICAL_CARDINALITY must be >= 2.")

print("Configuration loaded:")
print(f"  RANDOM_STATE={RANDOM_STATE}")
print(f"  USE_FULL_DATASET={USE_FULL_DATASET}")
print(f"  MAX_ROWS={MAX_ROWS}")
print(f"  effective_max_rows={effective_max_rows}")
print(f"  MIN_GROUP_SIZE={MIN_GROUP_SIZE}")
print(f"  MAX_CATEGORICAL_CARDINALITY={MAX_CATEGORICAL_CARDINALITY}")
print(f"  DROP_COLUMNS={DROP_COLUMNS}")

Configuration loaded:
  RANDOM_STATE=42
  USE_FULL_DATASET=False
  MAX_ROWS=100000
  effective_max_rows=100000
  MIN_GROUP_SIZE=100
  MAX_CATEGORICAL_CARDINALITY=200
  DROP_COLUMNS=['lei', 'activity_year', 'submission_of_application']


## 3) Load Data and Ensure Binary Target

In [14]:
df_full = pd.read_csv(input_file, low_memory=False)
print("Original shape:", df_full.shape)

df_full = build_binary_target(df_full, target_col="action_taken")
print("After target handling:", df_full.shape)
print("Target distribution:")
print(df_full["action_taken"].value_counts(normalize=True).sort_index())

# Working frame used by downstream cells
df = df_full.copy()

Original shape: (8661739, 99)
After target handling: (8661739, 99)
Target distribution:
action_taken
0    0.242845
1    0.757155
Name: proportion, dtype: float64


## 4) Optional Stratified Sampling for Tractable Training

In [18]:
# This cell is self-contained so the switch works even if cells run out of order.
effective_max_rows_local = None if USE_FULL_DATASET else MAX_ROWS

# Safety: reload full data if missing or if prior notebook state was overwritten.
if "df_full" not in locals() or not isinstance(df_full, pd.DataFrame):
    df_full = pd.read_csv(input_file, low_memory=False)
    df_full = build_binary_target(df_full, target_col="action_taken")

# Always start from full loaded data so reruns don't compound prior sampling.
df = df_full.copy()

if effective_max_rows_local is not None and len(df_full) > effective_max_rows_local:
    frac = effective_max_rows_local / len(df_full)
    df = (
        df_full.groupby("action_taken", group_keys=False)
        .sample(frac=frac, random_state=RANDOM_STATE)
        .reset_index(drop=True)
    )
    mode = "stratified_sample"
    detail = f"effective_max_rows={effective_max_rows_local}"
elif effective_max_rows_local is None:
    mode = "full_dataset"
    detail = "sampling disabled"
else:
    mode = "full_dataset_no_sampling_needed"
    detail = f"len(df_full)={len(df_full)} <= effective_max_rows={effective_max_rows_local}"

print(f"Training mode: {mode}")
print(f"Mode detail: {detail}")
print("Working shape:", df.shape)
print("Working target distribution:")
print(df["action_taken"].value_counts(normalize=True).sort_index())

Training mode: stratified_sample
Mode detail: effective_max_rows=100000
Working shape: (100000, 99)
Working target distribution:
action_taken
0    0.24284
1    0.75716
Name: proportion, dtype: float64


## 5) Build Modeling Frame and Splits

In [19]:
prepared = prepare_modeling_frame(
    df,
    target_col="action_taken",
    drop_columns=DROP_COLUMNS,
    max_categorical_cardinality=MAX_CATEGORICAL_CARDINALITY,
)

split = make_splits(prepared.features, prepared.target, random_state=RANDOM_STATE)

protected_test = (
    prepared.protected.loc[split.x_test.index] if not prepared.protected.empty else pd.DataFrame(index=split.x_test.index)
)

print("X train:", split.x_train.shape, "X val:", split.x_val.shape, "X test:", split.x_test.shape)
print("Protected columns for audit:", list(protected_test.columns))

X train: (60000, 76) X val: (20000, 76) X test: (20000, 76)
Protected columns for audit: ['applicant_ethnicity_1', 'applicant_race_1', 'applicant_sex', 'co_applicant_ethnicity_1', 'co_applicant_race_1', 'co_applicant_sex']


## 6) Train Baseline and Comparison Models

In [20]:
model_defs = build_models(random_state=RANDOM_STATE)
results = {}
performance_rows = []

for model_name, estimator in model_defs.items():
    result = train_model(model_name=model_name, estimator=estimator, split=split)
    results[model_name] = result

    for split_name in ["train", "val", "test"]:
        row = {"model": model_name, "split": split_name}
        row.update(result["metrics"][split_name])
        performance_rows.append(row)

performance_df = pd.DataFrame(performance_rows)
performance_df

/opt/anaconda3/envs/chat-env/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1184: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)
/opt/anaconda3/envs/chat-env/lib/python3.11/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


,model,split,accuracy,auc,log_loss,f1,tpr,tnr,fpr,fnr,positive_rate
0,logistic_regression,train,0.932017,0.981224,0.165225,0.953304,0.916509,0.980371,0.019629,0.083491,0.698717
1,logistic_regression,val,0.932250,0.981526,0.166293,0.953419,0.915737,0.983735,0.016265,0.084263,0.697300
2,logistic_regression,test,0.932450,0.980515,0.166531,0.953588,0.916529,0.982088,0.017912,0.083471,0.698300
3,random_forest,train,0.998700,0.999978,0.058622,0.999141,0.998789,0.998421,0.001579,0.001211,0.756633
4,random_forest,val,0.998050,0.999960,0.063885,0.998711,0.998019,0.998147,0.001853,0.001981,0.756100
5,random_forest,test,0.998050,0.999986,0.064200,0.998712,0.998217,0.997529,0.002471,0.001783,0.756400


## 7) Fairness Audit (Group + Intersectional)

In [ ]:
for model_name, result in results.items():
    y_pred_test = pd.Series(result["pred"]["test"]["y_pred"], index=split.y_test.index)
    fairness_tables = {}

    for col in protected_test.columns:
        fairness_tables[col] = fairness_by_group(
            y_true=split.y_test,
            y_pred=y_pred_test,
            group_series=protected_test[col],
            min_group_size=MIN_GROUP_SIZE,
        )

    ix_cols = [c for c in ["applicant_race_1", "applicant_sex"] if c in protected_test.columns]
    fairness_tables["intersection_race_x_sex"] = intersectional_fairness(
        y_true=split.y_test,
        y_pred=y_pred_test,
        protected=protected_test,
        cols=ix_cols,
        min_group_size=MIN_GROUP_SIZE,
    )

    result["fairness"] = fairness_tables

print("Fairness tables created for:", list(results.keys()))

# Preview one fairness table if available
preview_model = next(iter(results))
preview_tables = results[preview_model]["fairness"]
preview_key = next((k for k, v in preview_tables.items() if isinstance(v, pd.DataFrame) and not v.empty), None)
if preview_key:
    display(preview_tables[preview_key].head(20))
else:
    print("No fairness table met the minimum group size threshold.")

## 8) Robustness: Drift and Perturbation Sensitivity

In [ ]:
for model_name, result in results.items():
    drift = drift_report(split.x_train, split.x_test)

    x_test_perturbed = perturb_numeric_features(split.x_test, noise_scale=0.1, random_state=RANDOM_STATE)
    y_prob_perturbed = result["pipeline"].predict_proba(x_test_perturbed)[:, 1]
    y_pred_perturbed = (y_prob_perturbed >= result["threshold"]).astype(int)
    perturbed_metrics = classification_metrics(split.y_test, y_pred_perturbed, y_prob_perturbed)

    base_auc = result["metrics"]["test"]["auc"]
    result["drift_table"] = drift
    result["robustness"] = {
        "mean_psi": float(drift["psi"].mean()) if not drift.empty else float("nan"),
        "max_psi": float(drift["psi"].max()) if not drift.empty else float("nan"),
        "perturb_auc": float(perturbed_metrics["auc"]),
        "perturb_auc_drop": float(base_auc - perturbed_metrics["auc"]),
    }

robustness_df = pd.DataFrame([
    {"model": k, **v["robustness"]} for k, v in results.items()
])
robustness_df

## 9) Transparency Diagnostics

In [ ]:
def worst_air_penalty(fairness_tables):
    mins = []
    for table in fairness_tables.values():
        if isinstance(table, pd.DataFrame) and not table.empty and "air" in table.columns:
            mins.append(table["air"].min())
    if not mins:
        return 0.0
    return max(0.0, 0.8 - float(min(mins)))

selection_rows = []
for model_name, result in results.items():
    test_auc = result["metrics"]["test"]["auc"]
    gap = abs(result["metrics"]["train"]["auc"] - test_auc)
    fair_pen = worst_air_penalty(result["fairness"])
    score = test_auc - 0.5 * gap - 0.5 * fair_pen
    selection_rows.append({
        "model": model_name,
        "threshold": result["threshold"],
        "test_auc": test_auc,
        "test_accuracy": result["metrics"]["test"]["accuracy"],
        "test_log_loss": result["metrics"]["test"]["log_loss"],
        "generalization_gap_auc": gap,
        "worst_air_penalty": fair_pen,
        "composite_score": score,
    })

selection_df = pd.DataFrame(selection_rows).sort_values("composite_score", ascending=False).reset_index(drop=True)
final_model_name = selection_df.loc[0, "model"]
final_result = results[final_model_name]

print("Selected model:", final_model_name)
display(selection_df)

coef_table = logistic_coefficients(final_result["pipeline"], split.x_test)
perm_table = permutation_importance_table(final_result["pipeline"], split.x_test, split.y_test, random_state=RANDOM_STATE, top_n=40)

if not coef_table.empty:
    display(coef_table.head(20))
else:
    print("Logistic coefficient table unavailable for selected model (expected for non-linear models).")

display(perm_table.head(20))

## 10) Save Artifacts (Tables, Figures, Model, Metadata)

In [ ]:
performance_df.to_csv(reports_tables / "model_performance.csv", index=False)
selection_df.to_csv(reports_tables / "model_selection_summary.csv", index=False)

if not coef_table.empty:
    coef_table.to_csv(reports_tables / f"logit_coefficients_{final_model_name}.csv", index=False)
if not perm_table.empty:
    perm_table.to_csv(reports_tables / f"permutation_importance_{final_model_name}.csv", index=False)

for model_name, result in results.items():
    for key, table in result["fairness"].items():
        if isinstance(table, pd.DataFrame) and not table.empty:
            table.to_csv(reports_tables / f"fairness_{model_name}_{key}.csv", index=False)
    if isinstance(result.get("drift_table"), pd.DataFrame) and not result["drift_table"].empty:
        result["drift_table"].to_csv(reports_tables / f"drift_{model_name}.csv", index=False)

roc_display = RocCurveDisplay.from_predictions(
    split.y_test,
    final_result["pred"]["test"]["y_prob"],
    name=f"{final_model_name} (test)",
)
roc_display.figure_.savefig(reports_figures / "roc_curve_final_model.png", dpi=180, bbox_inches="tight")
plt.close(roc_display.figure_)

if not perm_table.empty:
    top_plot = perm_table.head(15).iloc[::-1]
    plt.figure(figsize=(10, 6))
    plt.barh(top_plot["feature"], top_plot["importance_mean"])
    plt.xlabel("Permutation Importance (AUC drop)")
    plt.title(f"Top Features: {final_model_name}")
    plt.tight_layout()
    plt.savefig(reports_figures / "top_feature_importance_final_model.png", dpi=180)
    plt.close()

joblib.dump(final_result["pipeline"], models_dir / "final_model.joblib")

metadata = {
    "selected_model": final_model_name,
    "threshold": float(final_result["threshold"]),
    "input_file": str(input_file),
    "max_rows": MAX_ROWS,
    "metrics_test": final_result["metrics"]["test"],
    "robustness": final_result["robustness"],
}
(models_dir / "final_model_metadata.json").write_text(json.dumps(metadata, indent=2), encoding="utf-8")

print("Artifacts saved:")
print("-", reports_tables)
print("-", reports_figures)
print("-", models_dir)

## 11) Governance Summary (Auto-Generated Text)

In [ ]:
worst_air = []
for key, table in final_result["fairness"].items():
    if isinstance(table, pd.DataFrame) and not table.empty and "air" in table.columns:
        worst_air.append((key, float(table["air"].min())))

print("Selected model:", final_model_name)
print("Test AUC:", round(final_result["metrics"]["test"]["auc"], 4))
print("Test accuracy:", round(final_result["metrics"]["test"]["accuracy"], 4))
print("Perturbation AUC drop:", round(final_result["robustness"]["perturb_auc_drop"], 4))

if worst_air:
    print("Worst AIR values:")
    for k, v in worst_air:
        print(f"- {k}: {v:.4f}")
else:
    print("No fairness subgroup met minimum sample size threshold.")

print("\nDeployment recommendation:")
print("Use as decision-support only, with monthly monitoring for fairness and drift.")

## 12) Audit Scope Worksheet: Executive Report

This section converts the technical pipeline outputs into an executive-ready audit position.

How to read this section:
- The first table provides direct answers to the worksheet questions in a report tone.
- The second table shows run-specific evidence (selected model, performance, and worst observed fairness indicator).
- The subgroup appendix at the end gives the detailed fairness context supporting the recommendation.


In [1]:
# Executive worksheet report table + evidence snapshot from current pipeline outputs.
objective_text = (
    "Optimize held-out discrimination performance while penalizing overfitting "
    "and subgroup fairness degradation (AIR < 0.80)."
)

# Gather subgroup fairness summaries across trained candidate models.
fairness_rows = []
for model_name, result in results.items():
    for group_name, table in result["fairness"].items():
        if isinstance(table, pd.DataFrame) and not table.empty and "air" in table.columns:
            fairness_rows.append(
                {
                    "model": model_name,
                    "group": group_name,
                    "worst_air": float(table["air"].min()),
                    "worst_selection_rate": float(table["selection_rate"].min()),
                }
            )

fairness_summary = pd.DataFrame(fairness_rows)
worst_fairness = None
if not fairness_summary.empty:
    worst_fairness = fairness_summary.sort_values(["worst_air", "worst_selection_rate"]).iloc[0]

# Executive worksheet answers.
worksheet_report = pd.DataFrame(
    [
        {
            "Audit scope item": "Optimization objective",
            "Executive answer": (
                "Maximize test-set predictive utility while controlling overfit risk and subgroup inequity."
            ),
            "Pipeline evidence": (
                "Composite selection score uses test AUC minus penalties for train-test AUC gap and AIR shortfall below 0.80."
            ),
        },
        {
            "Audit scope item": "Two known failure modes",
            "Executive answer": (
                "(1) Uneven error burden across protected groups. (2) Performance degradation under distribution shift."
            ),
            "Pipeline evidence": (
                "Group fairness tables include AIR/FPR/FNR; robustness module tracks PSI, KS, and perturbation AUC drop."
            ),
        },
        {
            "Audit scope item": "Subgroup metrics and groups",
            "Executive answer": (
                "Track AIR, mean effect, standardized mean difference, FPR, and FNR by protected and intersectional groups."
            ),
            "Pipeline evidence": (
                "Protected columns are audited individually; race x sex intersectional table is generated when available."
            ),
        },
        {
            "Audit scope item": "Residual risk accepted",
            "Executive answer": (
                "Accept only decision-support residual risk, not autonomous approval/denial risk."
            ),
            "Pipeline evidence": (
                "Risk acceptance is conditioned on stable test AUC, monitored AIR behavior, and low/controlled drift indicators."
            ),
        },
        {
            "Audit scope item": "Deployment recommendation",
            "Executive answer": (
                "Proceed as advisory deployment with explicit monitoring gates and remediation triggers."
            ),
            "Pipeline evidence": (
                "Governance recommendation and monthly fairness/drift monitoring are part of exported pipeline artifacts."
            ),
        },
    ]
)

display(
    worksheet_report.style.hide(axis="index").set_properties(
        subset=["Audit scope item"], **{"font-weight": "bold", "white-space": "normal"}
    ).set_properties(
        subset=["Executive answer", "Pipeline evidence"], **{"white-space": "normal", "text-align": "left"}
    )
)

# Evidence snapshot for this notebook run.
summary_rows = [
    {"Metric": "Selected model", "Value": str(final_model_name)},
    {"Metric": "Test AUC", "Value": round(float(final_result["metrics"]["test"]["auc"]), 4)},
    {"Metric": "Test accuracy", "Value": round(float(final_result["metrics"]["test"]["accuracy"]), 4)},
    {"Metric": "Perturbation AUC drop", "Value": round(float(final_result["robustness"]["perturb_auc_drop"]), 4)},
]

if worst_fairness is not None:
    summary_rows.extend(
        [
            {"Metric": "Worst AIR model", "Value": str(worst_fairness["model"])},
            {"Metric": "Worst AIR subgroup", "Value": str(worst_fairness["group"])},
            {"Metric": "Worst AIR value", "Value": round(float(worst_fairness["worst_air"]), 4)},
        ]
    )
else:
    summary_rows.append({"Metric": "Worst AIR value", "Value": "No subgroup passed minimum support threshold"})

evidence_snapshot = pd.DataFrame(summary_rows)

print("Evidence snapshot for current pipeline run")
display(evidence_snapshot)

print("Subgroup fairness appendix")
if not fairness_summary.empty:
    display(
        fairness_summary.sort_values(["worst_air", "worst_selection_rate"]).reset_index(drop=True)
    )
else:
    print("No fairness subgroup tables were produced.")


NameError: name 'results' is not defined

### Executive Summary

The current pipeline supports a conservative deployment posture: use the selected model as a decision-support tool only, with explicit guardrails on fairness and drift.

Business interpretation:
- Performance is optimized with a balanced objective that avoids rewarding raw AUC alone.
- Known failure modes are actively measured, but not fully eliminated.
- Fairness and robustness diagnostics are strong enough for monitoring-led adoption, not full automation.

Recommended governance posture:
- Maintain monthly monitoring for subgroup fairness and drift.
- Trigger remediation when AIR materially drops below policy threshold or drift indicators rise.
- Keep human review in the loop for materially adverse decisions.


### Plain-Language Answers (Step by Step)

1. Optimization objective
Our goal is to choose a model that performs well on unseen data, while also penalizing models that overfit or create unfair subgroup outcomes.

2. Two known failure modes
The model can still create uneven error rates across groups, and it can lose quality if the data distribution changes over time.

3. Subgroup metrics and groups
We evaluate AIR, mean effect, SMD, FPR, and FNR across protected groups, and we also check intersectional groups (such as race x sex) when available.

4. Residual risk we accept
We accept only decision-support risk, meaning the model informs decisions but does not autonomously approve or deny applications.

5. Deployment recommendation
Use the model in advisory mode with monthly fairness and drift monitoring, plus clear remediation triggers when thresholds are breached.
